# NB9 — RF-DETR + FashionCLIP Core-7 Detection V1

Notebook smoke-test cho implementation trong `src/detection`. Mục tiêu không phải so sánh detector nữa; RF-DETR được dùng để lấy garment boxes, còn `coarse_category` được dự đoán trực tiếp bằng cosine similarity giữa **FashionCLIP image embedding** và bảy text prototypes Core-7.

Canonical flow:

```text
image -> RF-DETR -> crop -> FashionCLIP 512-d L2 -> Core-7 cosine -> scorer handoff
```

`master_category` không được suy diễn cho ảnh user.


## 1. Runtime

Notebook này tự clone đúng feature branch và cài dependency detection. Checkpoint RF-DETR và FashionCLIP sẽ được tải/cache bởi runtime.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "feat/detection-rfdetr-fashionclip-core7"
REPO_DIR = Path("/content/opisoverated")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

os.chdir(REPO_DIR)
current_branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
print(f"Current working directory: {os.getcwd()}")
print(f"Git branch: {current_branch}")


In [ ]:
# Install the branch's pinned detection runtime dependencies.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-detection.txt")], check=True)
print("Detection dependencies installed.")


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "detection" / "config.py").exists():
    raise RuntimeError(
        f"Detection implementation is missing in {REPO_ROOT}. "
        f"Make sure branch {BRANCH!r} is checked out by rerunning the clone/checkout cell."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "configs/detection_rfdetr_fashionclip_core7_v1.json"
OUTPUT_ROOT = REPO_ROOT / "outputs/detection_v1"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing detection config: {CONFIG_PATH}")
print("repo:", REPO_ROOT)
print("config:", CONFIG_PATH)


## 2. Chọn ảnh

Đặt một ảnh local trong repo hoặc trỏ `IMAGE_PATH` tới file của bạn. Không hard-code Google Drive path trong notebook.


In [ ]:
IMAGE_PATH = REPO_ROOT / "animage.jpg"
if not IMAGE_PATH.is_file():
    print(f"Set IMAGE_PATH to an existing image before running inference: {IMAGE_PATH}")


## 3. Load versioned config


In [ ]:
from src.detection import load_detection_config

config = load_detection_config(CONFIG_PATH)
print(config)


## 4. Run detection

RF-DETR labels chỉ filter garment object/part. Category cuối cùng do FashionCLIP zero-shot Core-7 quyết định.


In [ ]:
from src.detection import DetectionPipeline

if IMAGE_PATH.is_file():
    pipeline = DetectionPipeline(config)
    result, image = pipeline.run(IMAGE_PATH)
    print("accepted garments:", len(result.garments))
    for garment in result.garments:
        print(
            garment.candidate.detector_label,
            "->",
            garment.category.coarse_category,
            f"sim={garment.category.similarity:.4f}",
            f"margin={garment.category.margin:.4f}",
        )


## 5. Save crops + scorer inputs


In [ ]:
from src.detection.pipeline import save_detection_result

if IMAGE_PATH.is_file():
    run_dir = OUTPUT_ROOT / IMAGE_PATH.stem
    saved = save_detection_result(
        result,
        image,
        run_dir,
        scorer_min_items=config.scorer_min_items,
        scorer_max_items=config.scorer_max_items,
    )
    print(saved)


## 6. Inspect metadata

`detection_result.json` giữ detector confidence, box, Core-7 cosine và margin nhưng **không tạo `master_category` giả**.


In [ ]:
import json

if IMAGE_PATH.is_file():
    metadata = json.loads((run_dir / "detection_result.json").read_text(encoding="utf-8"))
    print(json.dumps(metadata, ensure_ascii=False, indent=2)[:8000])


## 7. Visualize crops


In [ ]:
if IMAGE_PATH.is_file():
    import matplotlib.pyplot as plt
    from PIL import Image

    crop_paths = sorted((run_dir / "crops").glob("*.jpg"))
    for crop_path in crop_paths:
        plt.figure(figsize=(3, 3))
        plt.imshow(Image.open(crop_path))
        plt.title(crop_path.stem)
        plt.axis("off")
        plt.show()


## Acceptance notes

Smoke-test thành công chỉ xác nhận wiring/runtime. Trước production cần labeled validation cho detector box quality và Core-7 classification accuracy; xem `docs/DETECTION_CONTRACT_V1.md`.
